In [ ]:
pip install "transformers<4.45.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 113.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.10.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
pip install "numpy<2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 97.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy

In [ ]:
pip install sympy==1.13.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 45.0 MB/s eta 0:00:00
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.10.0+cu128 requires sympy>=1.13.3, but you have sympy 1.13.1 which is incompatible.


In [ ]:
import os
import gzip
import gc
import re
import pickle

from io import StringIO
from google.cloud import storage
from dotenv import load_dotenv

import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer, EsmForMaskedLM
from tokenizers import Tokenizer
from peft import get_peft_model, LoraConfig, TaskType

In [ ]:
import random
from scipy.stats import spearmanr

In [ ]:
import peft
print(peft.__version__)

0.18.1


In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.backends.cudnn.enabled)
print(torch.cuda.is_available())

2.10.0+cu128
12.8
True
True


In [ ]:
def initialize_progen2_noeval(model_name):
    '''
    Initializes the ProGen2 model with the given name.
    '''
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)


    return model, tokenizer

def collect_log_prob_pg2(sequence, model, tokenizer, device="cpu"):
    '''
    Creates a log probability matrix for each position in the protein
    for the protein with given sequence, using the given ProGen2 model
    and tokenizer.  Device is by default cpu but can be changed if using
    GPU or other device.  Outputs log probability matrix, reference log
    probability matrix and log loss ratio matrix.
    '''
    # Define indices for log-likelihood ratio matrix
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]

    prompt1 = "1"+sequence  # run it forwards
    prompt2 = "2"+sequence[::-1]  # run it backwards

    input_ids1 = torch.tensor(tokenizer.encode(prompt1)).unsqueeze(0).to(model.device)
    with torch.no_grad():
        logits1 = model(input_ids1).logits
    shift_logits1 = logits1[:, :-1, :]  # remove last entry

    input_ids2 =  torch.tensor(tokenizer.encode(prompt2)).unsqueeze(0).to(model.device)
    with torch.no_grad():
        logits2 = model(input_ids2).logits
    shift_logits2 = logits2[:, :-1, :] # remove last entry

    shift_logits2 = shift_logits2[:, torch.arange(shift_logits2.size(1) - 1, -1, -1), :]

    input_ids = input_ids1[:, 1:]

    logits = (shift_logits1 + shift_logits2)/2

    log_probs = F.log_softmax(logits, dim = -1)
    # n = log_probs.size(1)

    ref_log_probs = log_probs[0, torch.arange(input_ids.size(1)), input_ids[0]]
    ref_log_probs = ref_log_probs.unsqueeze(1)
    #ref_log_probs = ref_log_probs[:n-1]

    #log_probs = log_probs[0,:n-1]

    llr_matrix = log_probs - ref_log_probs
    llr_matrix = llr_matrix[0][:, aa_token_ids]
    log_probs = log_probs[0][:, aa_token_ids]

    return np.array(log_probs.cpu().detach()), np.array(ref_log_probs.cpu().detach()), np.array(llr_matrix.cpu().detach())

def insert_wt(seq, pos, wt_aa):
    seq_list = list(seq)
    pos = int(pos)
    if pos < len(seq_list):
        seq_list[pos] = wt_aa
    return ''.join(seq_list)

# computes the ranking loss between two iterables
def listwise_ranking_loss(preds, targets):
    indices = targets.sort(descending=True).indices
    preds = torch.gather(preds, dim=-1, index=indices)
    cumsums = preds.exp().flip(dims=[-1]).cumsum(dim=-1).flip(dims=[-1])
    loss = torch.log(cumsums + 1e-10) - preds
    return loss.mean()

def FineTune_ProGen2_LORA(device, base_model, tokenizer, lora_config,
                          protein_seq, dom_seq, dom_pos, target_tensor, loss_fn,
                          lrate=-1e-5, num_epochs=5, k=0.8,
                          num_samples=20, print_info=True):
    '''
    device is GPU or CPU
    base_model is ProGen2 model
    tokenizer is ProGen2 tokenizer
    lora_config is data for peft lora fine tuning
    lr is learning rate for gradient descent on new layer
    protein sequence is the sequence lora layer is being trained on
    target_tensor is what loss is measured against initially this is the actual protein sequence
    loss_fn is loss function used in training and validation
    num_epochs is the number of training steps
    k is the proportion of sequence used in training. validation and test indices are created as half the remaining indices each
    num_samples is the number of sample drawn for each epoch used for training and validation
    '''
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]

    model = get_peft_model(base_model, lora_config)

    model.to(device)

    ##############  new gemini fix
    for block in model.transformer.h:
        if hasattr(block.attn, 'scale_attn') and block.attn.scale_attn.device != device:
            block.attn.scale_attn = block.attn.scale_attn.to(device)
    ##############

    optimizer = torch.optim.AdamW(model.parameters(), lr=lrate)

    exp_tensor = target_tensor
    seq_len = exp_tensor.shape[0]

    all_indices = torch.randperm(seq_len)

    num_train = int(seq_len * k)

    train_indices = all_indices[:num_train]
    val_indices = all_indices[num_train:]

    train_losses = []
    val_losses = []
    early_stop_count = 0


    vocab_dict = tokenizer.get_vocab()
    seq_list = list(protein_seq)
    inputs = tokenizer(protein_seq, return_tensors="pt").to(device)

    for epoch in range(num_epochs):

        model.train()
        outputs = model(**inputs)
        logits = outputs.logits.squeeze(0)  # (seq_len, vocab_size)
        wt_logits = torch.log_softmax(logits, dim=-1)
        residue_indices = torch.arange(len(protein_seq))

        # ignoring BOS token
        seq_indices = [vocab_dict[aa] for aa in seq_list]
        wt_norm_tensor = wt_logits[residue_indices, seq_indices].unsqueeze(-1)
        LLR_tensor = wt_logits - wt_norm_tensor
        LLR_tensor_aa_only = LLR_tensor[:, aa_token_ids]
        LLR_tensor_domain = LLR_tensor_aa_only[dom_pos-1:dom_pos-1+len(dom_seq),:]

        # flatten the LLR_tensor
        # flattened_LLR_tensor = LLR_tensor_aa_only.flatten()

        flattened_LLR_tensor = LLR_tensor_domain.flatten()
        flattened_exp_tensor = exp_tensor.to(device)
        flattened_LLR_tensor = flattened_LLR_tensor.to(device)

        #to stack
        combined = torch.stack([flattened_LLR_tensor, flattened_exp_tensor], dim=0)
        ft_tensor = torch.transpose(combined, 0, 1)

        # predicted_scores = []
        # experimental_values = []

        train_tensor = ft_tensor[train_indices]

        #drop nan
        train_tensor = train_tensor[~torch.any(train_tensor.isnan(), dim=1)]

        # num_samples = num_samples
        positions = train_tensor[torch.randperm(len(train_tensor))[:num_samples]]

        predicts = positions[:, 0] #LLR
        targets = positions[:, 1] #exp

        # Compute loss with predicts and targets
        loss = loss_fn(predicts, targets)
        # log_probs = torch.log_softmax(logits, dim=-1)
        # loss = -log_probs[torch.arange(len(seq_indices)), seq_indices].mean() # needs to be difference of predict/target
        loss.backward()
        # for name, param in model.named_parameters():
        #     if param.requires_grad:
        #         print(name, param.grad.abs().mean())
        optimizer.step()
        optimizer.zero_grad()

        train_losses.append(loss.item())

        # ----------- VALIDATION (no backprop) -----------
        model.eval()
        with torch.no_grad():
            val_tensor = ft_tensor[val_indices]
            val_tensor = val_tensor [~torch.any(val_tensor.isnan(), dim=1)]

            # num_samples = 45
            positions = val_tensor[torch.randperm(len(val_tensor))[:num_samples]]

            predicts = positions[:, 0] #LLR
            targets = positions[:, 1] #exp

            val_loss = loss_fn(predicts, targets)
            # val_loss = listwise_ranking_loss(predicts, targets)
            #   val_log_probs = torch.log_softmax(logits, dim=-1)
            #   val_loss = -log_probs[torch.arange(len(seq_indices)), seq_indices].mean()
            val_losses.append(val_loss.item())

        if print_info==True:
            print(f"Epoch {epoch+1} - Training Loss: {loss.item():.4f} | Validation Loss: {val_loss.item():.4f}")

            if val_loss.item() > loss.item():
                early_stop_count += 1
            else:
                early_stop_count = 0

            print(f"Validation loss has exceeded training loss {early_stop_count} time(s) in a row")
            # if early_stop_count > 2:
            #     print("Validation loss exceeded training loss 3 times — early stopping.")
            #     break

    return model, train_losses, val_losses
# , ft_tensor, test_indices

def spearman_ignore_nan(a, b):
    mask = ~np.isnan(a) & ~np.isnan(b)
    return spearmanr(a[mask], b[mask])

In [ ]:
# add features to choose best number of epochs less than or equal to num_epochs
# add spearman correlation as metric using fitness
# output spearman correlation
# choose best number of epochs based on val_loss or spearman max

def Opt_FineTune_ProGen2_LORA(device, base_model, tokenizer, lora_config,
                          protein_seq, dom_seq, dom_pos, target_tensor, loss_fn,
                          lrate=-1e-5, num_epochs=5, k=0.8,
                          num_samples=20, print_info=True):
    '''
    device is GPU or CPU
    base_model is ProGen2 model
    tokenizer is ProGen2 tokenizer
    lora_config is data for peft lora fine tuning
    lr is learning rate for gradient descent on new layer
    protein sequence is the sequence lora layer is being trained on
    target_tensor is what loss is measured against initially this is the actual protein sequence
    loss_fn is loss function used in training and validation
    num_epochs is the number of training steps
    k is the proportion of sequence used in training. validation and test indices are created as half the remaining indices each
    num_samples is the number of sample drawn for each epoch used for training and validation
    '''
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]

    model = get_peft_model(base_model, lora_config)
    model.to(device)



    ##############  new gemini fix
    for block in model.transformer.h:
        if hasattr(block.attn, 'scale_attn') and block.attn.scale_attn.device != device:
            block.attn.scale_attn = block.attn.scale_attn.to(device)
    ##############

    optimizer = torch.optim.AdamW(model.parameters(), lr=lrate)

    exp_tensor = target_tensor
    seq_len = exp_tensor.shape[0]

    all_indices = torch.randperm(seq_len)

    num_train = int(seq_len * k)

    train_indices = all_indices[:num_train]
    val_indices = all_indices[num_train:]

    train_losses = []
    val_losses = []
    val_rhos = []
    num_eps = 0
    early_stop_count = 0

    best_val_loss = float("inf")
    best_val_rho  = -float("inf")
    best_lora_state = None

    vocab_dict = tokenizer.get_vocab()
    seq_list = list(protein_seq)
    inputs = tokenizer(protein_seq, return_tensors="pt").to(device)


    for epoch in range(1, num_epochs+1):

        model.train()
        outputs = model(**inputs)
        logits = outputs.logits.squeeze(0)  # (seq_len, vocab_size)
        wt_logits = torch.log_softmax(logits, dim=-1)
        residue_indices = torch.arange(len(protein_seq))

        # ignoring BOS token
        seq_indices = [vocab_dict[aa] for aa in seq_list]
        wt_norm_tensor = wt_logits[residue_indices, seq_indices].unsqueeze(-1)
        LLR_tensor = wt_logits - wt_norm_tensor
        LLR_tensor_aa_only = LLR_tensor[:, aa_token_ids]
        LLR_tensor_domain = LLR_tensor_aa_only[dom_pos-1:dom_pos-1+len(dom_seq),:]

        # flatten the LLR_tensor
        # flattened_LLR_tensor = LLR_tensor_aa_only.flatten()

        flattened_LLR_tensor = LLR_tensor_domain.flatten()
        flattened_exp_tensor = exp_tensor.to(device)
        flattened_LLR_tensor = flattened_LLR_tensor.to(device)

        #to stack
        combined = torch.stack([flattened_LLR_tensor, flattened_exp_tensor], dim=0)
        ft_tensor = torch.transpose(combined, 0, 1)

        # predicted_scores = []
        # experimental_values = []

        train_tensor = ft_tensor[train_indices]

        #drop nan
        train_tensor = train_tensor[~torch.any(train_tensor.isnan(), dim=1)]

        # num_samples = num_samples
        positions = train_tensor[torch.randperm(len(train_tensor))[:num_samples]]

        predicts = positions[:, 0] #LLR
        targets = positions[:, 1] #exp

        # Compute loss with predicts and targets
        loss = loss_fn(predicts, targets)
        # log_probs = torch.log_softmax(logits, dim=-1)
        # loss = -log_probs[torch.arange(len(seq_indices)), seq_indices].mean() # needs to be difference of predict/target
        loss.backward()
        # for name, param in model.named_parameters():
        #     if param.requires_grad:
        #         print(name, param.grad.abs().mean())
        optimizer.step()
        optimizer.zero_grad()

        train_losses.append(loss.item())

        # ----------- VALIDATION (no backprop) -----------
        model.eval()
        with torch.no_grad():
            val_tensor = ft_tensor[val_indices]
            val_tensor = val_tensor [~torch.any(val_tensor.isnan(), dim=1)]

            positions = val_tensor[torch.randperm(len(val_tensor))[:num_samples]]

            predicts = positions[:, 0] #LLR
            targets = positions[:, 1] #exp

            val_loss = loss_fn(predicts, targets)
            val_losses.append(val_loss.item())

            val_rho, _ = spearman_ignore_nan(predicts.cpu().numpy(), targets.cpu().numpy())
            val_rhos.append(val_rho)

        if val_loss < best_val_loss or val_rho > best_val_rho:
            num_eps = epoch
            best_val_loss = min(val_loss, best_val_loss)
            best_val_rho  = max(val_rho, best_val_rho)
            # Save only LoRA weights — ~10-50MB instead of ~2.5GB
            best_lora_state = {k: v.cpu().clone()
                               for k, v in model.state_dict().items()
                               if "lora_" in k}

            print(f"*Best val loss = {best_val_loss}*")
            print(f"*Best val rho = {best_val_rho}*")

        if len(val_losses) >= 3:
            loss_worse = val_losses[-1] > val_losses[-2] and val_losses[-2] > val_losses[-3]
            rho_worse  = val_rhos[-1]   < val_rhos[-2]   and val_rhos[-2]   < val_rhos[-3]
            if loss_worse or rho_worse:
                break


        if print_info==True:
            print(f"Epoch {epoch+1} - Training Loss: {loss.item():.4f} | Validation Loss: {val_loss.item():.4f}")

            if val_loss.item() > loss.item():
                early_stop_count += 1
            else:
                early_stop_count = 0

            print(f"Validation loss has exceeded training loss {early_stop_count} time(s) in a row")
            # if early_stop_count > 2:
            #     print("Validation loss exceeded training loss 3 times — early stopping.")
            #     break

    if model is not None:

        base_model = model.unload()

        if hasattr(base_model, "peft_config"):
            del base_model.peft_config

        del model

    gc.collect()
    torch.cuda.empty_cache()

    train_losses = [round(x, 4) for x in train_losses]
    val_losses = [round(x, 4) for x in val_losses]
    val_rhos = [round(x, 4) for x in val_rhos]

    return best_lora_state, train_losses, val_losses, val_rhos, num_eps


# GCP and model loading

In [ ]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] ='plm-study-484223-e1c13b49d132.json'

# device = 'cpu'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using {device} device")
model_name = "hugohrban/progen2-medium"
base_model, tokenizer = initialize_progen2_noeval(model_name)

# gcs information
client = storage.Client()
bucket = client.bucket('domainome-data')

# load fitness data filtered for ProGen2 context window of 1024
blob_name = 'dict_dn_fitness_filtered.pkl'
blob = bucket.blob(blob_name)
data_bytes = blob.download_as_bytes()
dict_dn_fitness_filtered = pickle.loads(data_bytes)

# load domainome data
blob_name = 'dict_domainome_uniprot_new.pkl'
blob = bucket.blob(blob_name)
data_bytes = blob.download_as_bytes()
dict_uniprot = pickle.loads(data_bytes)

Using cuda device


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

The repository for hugohrban/progen2-medium contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/hugohrban/progen2-medium.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


configuration_progen.py: 0.00B [00:00, ?B/s]

...

The repository for hugohrban/progen2-medium contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/hugohrban/progen2-medium.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y
The repository for hugohrban/progen2-medium contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/hugohrban/progen2-medium.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


modeling_progen.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/hugohrban/progen2-medium:
- modeling_progen.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

# Loop for checking hyperparameters on validation set and saving best model and best model output.

In [ ]:
from itertools import product
import datetime

In [ ]:
# loop that saves LLR outputs from lora models as pkl.gz files of dictionaries
# today = datetime.date.today()
# formatted_date = today.strftime("%Y-%m-%d")
formatted_date = '40_' + '2026-04-09'

keys = list(dict_dn_fitness_filtered.keys())

loss = listwise_ranking_loss

eps = 20
ranks = [16, 32, 64, 128]
learning_rates = [1e-4, 5e-4, 1e-3, 5e-3]
num_samples = 40

hypers = list(product(ranks, learning_rates))

#loop to create models
for i in range(451,501):
    domain_id = keys[i]
    print(f"{i} -- {domain_id}")
    dict_entry = dict_dn_fitness_filtered[domain_id]
    dom_pos = int(dict_entry['domain_start'])
    dom_seq = dict_entry['dom_seq']

    uniprot_id = dict_entry['uniprot_id']
    protein_seq = dict_uniprot[uniprot_id]['sequence'][:1023]

    fitness_list = dict_entry['fitness']
    fitness_tensor = torch.tensor(fitness_list)

    best_best_val_rho = -float("inf")
    best_best_val_loss = float("inf")

    for rank, lr in hypers:

        rank = int(rank)
        alpha = 2*rank
        print("---------------------------------")
        print(f"rank = {rank}, lr = {lr}")

        lora_config = LoraConfig(
        r=rank,
        lora_alpha=alpha,
        target_modules=["qkv_proj", "out_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM)

        lora_weights, train_losses, val_losses, val_rhos, num_eps = Opt_FineTune_ProGen2_LORA(device, base_model, tokenizer,
                                                                            lora_config, protein_seq, dom_seq, dom_pos,
                                                                            fitness_tensor, loss, lrate=lr, num_epochs=eps,
                                                                            k=0.8, num_samples=num_samples, print_info=False)
        print("---------------------------------")
        print(f"Number of epochs = {num_eps}")
        print(f"Training loss = {train_losses}")
        print(f"Validation loss = {val_losses}")
        print(f"Spearman score = {val_rhos}")

        print(f"val_rho for {num_eps} eps = {val_rhos[num_eps-1]}")
        print(f"val_loss for {num_eps} eps = {val_losses[num_eps - 1]}")
        print("---------------------------------")

        # if val_rhos[num_eps-1] > best_best_val_rho or val_losses[num_eps-1] < best_best_val_loss:
        if val_rhos[num_eps-1] > best_best_val_rho:
            best_best_val_rho = max(val_rhos[num_eps-1], best_best_val_rho)
            best_best_val_loss = min(val_losses[num_eps-1], best_best_val_loss)

            best_lr = lr
            best_rank = rank
            best_alpha = 2*best_rank
            best_val_losses = val_losses
            best_train_losses = train_losses
            best_val_rhos = val_rhos
            best_num_eps = num_eps
            best_lora_weights = lora_weights

            print(f"best_best_val_rho = {best_best_val_rho}")
            print(f"best_best_val_loss = {best_best_val_loss}")

    print('======================Model Selected==============================')
    print(f"Best model lr = {best_lr}")
    print(f"Best model rank = {best_rank}")
    print(f"Best model num eps = {best_num_eps}")
    print(f"Best model train loss = {best_train_losses}")
    print(f"Best model val losses = {best_val_losses}")
    print(f"Best model val rhos = {best_val_rhos}")
    print("------------------------------------------------------------------")
    print(f"Val loss for num_eps is {best_val_losses[best_num_eps - 1]}")
    print(f"Val rho for num_eps is {best_val_rhos[best_num_eps - 1]}")
    print('====================Creating Output Dictionary====================')



    best_lora_config = LoraConfig(
        r=best_rank,
        lora_alpha=best_alpha,
        target_modules=["qkv_proj", "out_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM)


    best_model = get_peft_model(base_model, best_lora_config)
    best_model.load_state_dict(best_lora_weights, strict=False)
    best_model.to(device)
    best_model.eval()

    # new dictionary to store fine-tuned LLR
    dict_uniprot_LLRs = dict_uniprot.copy()

    with torch.no_grad():
        for key in dict_uniprot_LLRs.keys():

            seq = dict_uniprot_LLRs[key]['sequence']
            # print(len(seq))
            if len(seq) > 1024:
                seq = seq[:1023]
            # print(len(seq))
            lp, rlp, llr = collect_log_prob_pg2(seq, best_model, tokenizer)
            dict_uniprot_LLRs[key]['LLR'] = llr

    # optimal metrics for this run
    dict_uniprot_LLRs['eps'] = best_num_eps
    dict_uniprot_LLRs['lr'] = best_lr
    dict_uniprot_LLRs['num_samples'] = num_samples
    dict_uniprot_LLRs["train"] = best_train_losses
    dict_uniprot_LLRs["val"] = best_val_losses
    dict_uniprot_LLRs["rho"] = best_val_rhos
    dict_uniprot_LLRs["rank"] = best_rank

    print('====================Output Dictionary Saved=======================')

    # store output of best model in gcp bucket
    data_out = dict_uniprot_LLRs
    compressed_data_out = gzip.compress(pickle.dumps(data_out))

    blob_name = domain_id+"_lora_LLRs.pkl.gz"
    folder_name = formatted_date+"_lora_dicts_full/"
    blob = bucket.blob(folder_name + blob_name)

    blob.upload_from_string(compressed_data_out)

    # store best model state in gcp bucket
    data_model = {
                  'lora_config': best_lora_config,
                  'lora_weights': best_lora_weights
                  }
    compressed_data_model = gzip.compress(pickle.dumps(data_model))

    blob_name = domain_id+"_lora_model.pkl.gz"
    folder_name = formatted_date+"_lora_models_full/"
    blob = bucket.blob(folder_name + blob_name)

    blob.upload_from_string(compressed_data_model)

    print(f'====================Model {blob_name} Saved======================')
    print()

    if best_model is not None:

        base_model = best_model.unload()

        if hasattr(base_model, "peft_config"):
            del base_model.peft_config

        del best_model

    gc.collect()
    torch.cuda.empty_cache()

451 -- Q9UQR1_PF00096_172
---------------------------------
rank = 16, lr = 0.0001
*Best val loss = 11.503803253173828*
*Best val rho = -0.35778611632270174*
*Best val loss = 6.750840663909912*
*Best val rho = -0.35778611632270174*
*Best val loss = 5.818219184875488*
*Best val rho = -0.2915572232645404*
*Best val loss = 5.463898658752441*
*Best val rho = -0.2031894934333959*
*Best val loss = 5.32785177230835*
*Best val rho = -0.17016885553470923*
*Best val loss = 5.202737808227539*
*Best val rho = -0.17016885553470923*
*Best val loss = 5.202737808227539*
*Best val rho = -0.025328330206378986*
*Best val loss = 3.92820143699646*
*Best val rho = -0.025328330206378986*
*Best val loss = 3.92820143699646*
*Best val rho = 0.09174484052532834*
---------------------------------
Number of epochs = 10
Training loss = [8.0407, 4.6672, 6.0016, 5.6742, 4.7253, 5.0828, 6.351, 3.6313, 3.8465, 3.9595, 5.1454]
Validation loss = [11.5038, 6.7508, 10.4105, 5.8182, 5.4639, 5.3279, 5.2027, 5.9018, 3.9282, 4

IndexError: list index out of range